In [ ]:
%load_ext autoreload
%autoreload 2
import sys, json, time, gc, warnings
from pathlib import Path
import numpy as np
import polars as pl
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore', message='Overriding the parameters from Reference Dataset')

ROOT = Path.cwd().resolve()
while not ((ROOT / 'src').is_dir() and (ROOT / 'configs').is_dir()):
    if ROOT.parent == ROOT:
        raise FileNotFoundError('Không tìm thấy thư mục gốc chứa src/ và configs/')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import load_config
from src.features.prepare import META, AUDIT_COLS
from src.features import encode as E

cfg = load_config()
PROC, REPORTS, MODELS = ROOT / 'data' / 'processed', ROOT / 'reports', ROOT / 'models'
assert 'person_1__app__gender_992L' in AUDIT_COLS, 'Chưa sửa AUDIT_COLS trong prepare.py'

# ---- Cửa sổ & tham số (giữ như notebook 07) ----
FIT_WEEKS, ES_WEEKS, VALID_WEEKS = (0, 39), (40, 43), (44, 51)
SEEDS = [42, 43, 44]
N_JOBS = 11
NUM_BOOST_ROUND, EARLY_STOPPING = 5000, 200
LGB_PARAMS = {
    'objective': 'binary', 'metric': 'auc', 'learning_rate': 0.05, 'num_leaves': 63,
    'min_child_samples': 200, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 1,
    'lambda_l2': 10.0, 'deterministic': True, 'force_row_wise': True, 'num_threads': N_JOBS, 'verbosity': -1,
}

def gini_stability(week, y_true, y_score):
    week, y_true, y_score = map(np.asarray, (week, y_true, y_score))
    weeks = np.unique(week)
    ginis = np.array([2 * roc_auc_score(y_true[week == w], y_score[week == w]) - 1 for w in weeks])
    a, b = np.polyfit(weeks.astype(float), ginis, 1)
    residuals = ginis - (a * weeks + b)
    return {'gini_overall': float(2 * roc_auc_score(y_true, y_score) - 1),
            'mean_gini': float(ginis.mean()), 'slope_a': float(a), 'std_resid': float(residuals.std()),
            'stability': float(ginis.mean() + 88.0 * min(0.0, a) - 0.5 * residuals.std()),
            'weekly': dict(zip(weeks.tolist(), ginis.round(4).tolist()))}

# ---- Nạp tuần ≤ 51 ----
t0 = time.time()
lf = pl.scan_parquet(PROC / 'model_table.parquet')
schema = lf.collect_schema()
FEATS = [c for c in schema.names() if c not in set(META) | set(AUDIT_COLS)]
STR_FEATS = [c for c in FEATS if schema[c] == pl.String]
assert len(FEATS) == 626, len(FEATS)
df = lf.filter(pl.col('WEEK_NUM') <= VALID_WEEKS[1]).select(['case_id', 'WEEK_NUM', 'target'] + FEATS).collect(engine='streaming')

# ---- Encoder (R8) fit trên tuần 0–39 ----
ENC = E.fit_encoder(df.select(['WEEK_NUM'] + STR_FEATS).filter(pl.col('WEEK_NUM').is_between(*FIT_WEEKS)).drop('WEEK_NUM'), FEATS)
ENC['n_train'] = int(df.filter(pl.col('WEEK_NUM').is_between(*FIT_WEEKS)).height)
E.save_encoder(ENC, REPORTS / 'enc_r7_w0_39.json')

def part(lo, hi):
    p = df.filter(pl.col('WEEK_NUM').is_between(lo, hi))
    return (E.transform(p, ENC), p['target'].to_numpy().astype(np.int8),
            p['WEEK_NUM'].to_numpy(), p['case_id'].to_numpy())

X_fit, y_fit, w_fit, id_fit = part(*FIT_WEEKS)
X_es,  y_es,  w_es,  id_es  = part(*ES_WEEKS)
X_val, y_val, w_val, id_val = part(*VALID_WEEKS)
del df; gc.collect()

FEAT_NAMES, CAT_COLS = ENC['out_feats'], ENC['cat_cols']
print(f'FEATS gốc: {len(FEATS)} | String: {len(STR_FEATS)} | out_feats: {len(FEAT_NAMES)} | '
      f"cat_cols: {len(CAT_COLS)} | hc: {len(ENC['hc_cols'])} | n_train: {ENC['n_train']:,}")
for name, X, y, w in [('fit', X_fit, y_fit, w_fit), ('es', X_es, y_es, w_es), ('valid', X_val, y_val, w_val)]:
    print(f'{name:5s} {X.shape} | dr {y.mean():.4f} | tuần {w.min()}–{w.max()} | {X.nbytes/1e9:.2f} GB')
same_order = np.array_equal(id_val, pl.read_parquet(REPORTS / 'lgb07_valid_preds.parquet')['case_id'].to_numpy())
print(f'Thứ tự Valid trùng notebook 07: {same_order} | {time.time()-t0:.0f}s')

In [ ]:
def run_lgb(tag, Xf, Xe, Xv, names, cats, override=None, base_pval=None, seeds=SEEDS, save_model=True):
    override = override or {}
    dtr = lgb.Dataset(Xf, label=y_fit, feature_name=names, categorical_feature=cats,
                      params={'data_random_seed': 42, 'verbosity': -1}, free_raw_data=False)
    des = lgb.Dataset(Xe, label=y_es, reference=dtr, free_raw_data=False)
    out, P, IMP = {'tag': tag, 'override': override, 'n_feats': len(names), 'seeds': {}}, {}, []
    for s in seeds:
        t0 = time.time()
        params = {**LGB_PARAMS, **override, 'bagging_seed': s, 'feature_fraction_seed': s}
        b = lgb.train(params, dtr, num_boost_round=NUM_BOOST_ROUND, valid_sets=[des], valid_names=['es'],
                      callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False)])
        bi = b.best_iteration
        p = b.predict(Xv, num_iteration=bi)
        m = gini_stability(w_val, y_val, p)
        gfit = float(2 * roc_auc_score(y_fit, b.predict(Xf, num_iteration=bi)) - 1)
        out['seeds'][s] = {'best_iter': int(bi), 'gini_fit': gfit, 'gap_fit_valid': gfit - m['gini_overall'],
                           **m, 'time_s': round(time.time() - t0, 1)}
        P[s] = p
        IMP.append(b.feature_importance('gain', iteration=bi))
        np.save(REPORTS / f'lgb08_{tag}_pval_s{s}.npy', p)
        if save_model:
            b.save_model(str(MODELS / f'lgb08_{tag}_s{s}.txt'), num_iteration=bi)
        r = out['seeds'][s]
        print(f"[{tag}] seed {s} | iter {bi} | fit {gfit:.4f} | Valid {r['gini_overall']:.4f} | "
              f"gap {r['gap_fit_valid']:.4f} | stab {r['stability']:.4f} | {r['time_s']:.0f}s")
        del b; gc.collect()

    p_ens = np.mean([P[s] for s in seeds], axis=0)
    out['ensemble'] = gini_stability(w_val, y_val, p_ens)
    for k in ['gini_overall', 'stability', 'gap_fit_valid', 'best_iter']:
        v = np.array([out['seeds'][s][k] for s in seeds], dtype=float)
        out[f'{k}_mean_std'] = (float(v.mean()), float(v.std(ddof=1)))
    if base_pval is not None:
        mb = gini_stability(w_val, y_val, base_pval)
        dw = {w: out['ensemble']['weekly'][w] - mb['weekly'][w] for w in mb['weekly']}
        out['vs_base'] = {'d_valid_ens': out['ensemble']['gini_overall'] - mb['gini_overall'],
                          'd_stab_ens': out['ensemble']['stability'] - mb['stability'],
                          'weeks_won': int(sum(v > 0 for v in dw.values())),
                          'weekly_delta': {w: round(v, 4) for w, v in dw.items()}}
    g = np.mean([x / x.sum() for x in IMP], axis=0)
    pl.DataFrame({'feature': names, 'gain_share': g}).sort('gain_share', descending=True) \
      .write_csv(REPORTS / f'lgb08_{tag}_importance.csv')
    with open(REPORTS / f'lgb08_{tag}.json', 'w', encoding='utf-8') as f:
        json.dump(out, f, indent=2, default=str)

    e = out['ensemble']
    print(f"\n[{tag}] per-seed Valid {out['gini_overall_mean_std'][0]:.5f} ± {out['gini_overall_mean_std'][1]:.5f} | "
          f"gap {out['gap_fit_valid_mean_std'][0]:.4f} | iter {out['best_iter_mean_std'][0]:.0f}")
    print(f"[{tag}] ensemble Valid {e['gini_overall']:.4f} | stab {e['stability']:.4f} "
          f"(mean {e['mean_gini']:.4f}, a {e['slope_a']:+.5f}, std_res {e['std_resid']:.4f})")
    if 'vs_base' in out:
        vb = out['vs_base']
        print(f"[{tag}] so mốc (ensemble): ΔValid {vb['d_valid_ens']:+.4f} | Δstab {vb['d_stab_ens']:+.4f} | "
              f"thắng {vb['weeks_won']}/8 | {vb['weekly_delta']}")
    return out, p_ens

EXP, PENS = {}, {}
P07_DFREQ = np.mean([np.load(REPORTS / f'lgb07_d_freq_pval_s{s}.npy') for s in SEEDS], axis=0)
EXP['ref'], PENS['ref'] = run_lgb('ref', X_fit, X_es, X_val, FEAT_NAMES, CAT_COLS, base_pval=P07_DFREQ)

pl.DataFrame({'case_id': id_val, 'WEEK_NUM': w_val, 'target': y_val, 'pred_ref': PENS['ref']}) \
  .write_parquet(REPORTS / 'lgb08_ref_valid_preds.parquet')
imp_ref = pl.read_csv(REPORTS / 'lgb08_ref_importance.csv')
print(f"\nGain = 0: {(imp_ref['gain_share'] == 0).sum()} / {len(FEAT_NAMES)} | top 50 chiếm {imp_ref['gain_share'][:50].sum():.3f}")
with pl.Config(fmt_str_lengths=60):
    print(imp_ref.head(15))

In [ ]:
HC = ENC['hc_cols']
TE_M, FOLD_W = 200, 8

hc_df = (pl.scan_parquet(PROC / 'model_table.parquet')
           .filter(pl.col('WEEK_NUM') <= VALID_WEEKS[1])
           .select(['case_id', 'WEEK_NUM', 'target'] + HC).collect(engine='streaming'))
fit_df = hc_df.filter(pl.col('WEEK_NUM').is_between(*FIT_WEEKS)).with_columns((pl.col('WEEK_NUM') // FOLD_W).alias('fold'))
es_df = hc_df.filter(pl.col('WEEK_NUM').is_between(*ES_WEEKS))
va_df = hc_df.filter(pl.col('WEEK_NUM').is_between(*VALID_WEEKS))
assert np.array_equal(fit_df['case_id'].to_numpy(), id_fit)
assert np.array_equal(es_df['case_id'].to_numpy(), id_es)
assert np.array_equal(va_df['case_id'].to_numpy(), id_val)
print('Fold (khối 8 tuần):', fit_df.group_by('fold').agg(pl.len(), pl.col('target').mean().round(4)).sort('fold').rows())

N, Y = fit_df.height, int(fit_df['target'].sum())
P0 = Y / N
fold_tot = fit_df.group_by('fold').agg(pl.len().alias('Nk'), pl.col('target').sum().alias('Yk'))

def te_oof_fit(c):
    tot = fit_df.group_by(c).agg(pl.len().alias('n'), pl.col('target').sum().alias('s')).drop_nulls(c)
    byf = fit_df.group_by([c, 'fold']).agg(pl.len().alias('nf'), pl.col('target').sum().alias('sf')).drop_nulls(c)
    f = (fit_df.select([c, 'fold']).with_row_index('i')
           .join(tot, on=c, how='left').join(byf, on=[c, 'fold'], how='left').join(fold_tot, on='fold', how='left')
           .with_columns(((Y - pl.col('Yk')) / (N - pl.col('Nk'))).alias('prior'))
           .with_columns(pl.when(pl.col(c).is_null()).then(None)
                           .otherwise((pl.col('s') - pl.col('sf') + TE_M * pl.col('prior'))
                                      / (pl.col('n') - pl.col('nf') + TE_M)).alias('te'))
           .sort('i'))
    return f['te'].cast(pl.Float32).to_numpy(), tot

def te_apply(d, c, tot):
    f = (d.select(c).with_row_index('i').join(tot, on=c, how='left')
           .with_columns(pl.when(pl.col(c).is_null()).then(None)
                           .otherwise((pl.col('s').fill_null(0) + TE_M * P0) / (pl.col('n').fill_null(0) + TE_M)).alias('te'))
           .sort('i'))
    return f['te'].cast(pl.Float32).to_numpy()

T_fit = np.empty((N, len(HC)), np.float32)
T_es = np.empty((es_df.height, len(HC)), np.float32)
T_val = np.empty((va_df.height, len(HC)), np.float32)
for k, c in enumerate(HC):
    T_fit[:, k], tot = te_oof_fit(c)
    T_es[:, k], T_val[:, k] = te_apply(es_df, c, tot), te_apply(va_df, c, tot)
TE_NAMES = [f'{c}__te' for c in HC]
del hc_df, fit_df, es_df, va_df; gc.collect()

# ---- Gini đơn biến trên Valid: __freq so với __te ----
def uni_gini(x, y):
    m = ~np.isnan(x)
    return 2 * roc_auc_score(y[m], x[m]) - 1 if m.sum() > 0 else np.nan
for k, c in enumerate(HC):
    jf = FEAT_NAMES.index(f'{c}__freq')
    print(f'{c:45s} freq {uni_gini(X_val[:, jf], y_val):+.4f} | te {uni_gini(T_val[:, k], y_val):+.4f} | '
          f'null Valid {np.isnan(T_val[:, k]).mean():.3f}')

# ---- Huấn luyện: ref + 7 cột TE ----
Xf_te, Xe_te, Xv_te = np.hstack([X_fit, T_fit]), np.hstack([X_es, T_es]), np.hstack([X_val, T_val])
EXP['te200'], PENS['te200'] = run_lgb('te200', Xf_te, Xe_te, Xv_te, FEAT_NAMES + TE_NAMES, CAT_COLS,
                                      base_pval=PENS['ref'])
imp = pl.read_csv(REPORTS / 'lgb08_te200_importance.csv').with_row_index('rank')
print(imp.filter(pl.col('feature').is_in(TE_NAMES)))

In [ ]:
try:
    import xgboost as xgb
except ImportError:
    raise ImportError('Chạy trong terminal (venv): pip install xgboost catboost')
from scipy.stats import spearmanr
print('xgboost', xgb.__version__)

FT_REF = ['c' if n in set(CAT_COLS) else 'q' for n in FEAT_NAMES]
XGB_PARAMS = {'objective': 'binary:logistic', 'eval_metric': 'auc', 'tree_method': 'hist',
              'max_depth': 6, 'eta': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8,
              'min_child_weight': 5, 'lambda': 10.0, 'max_bin': 256, 'nthread': N_JOBS, 'verbosity': 0}

def finish(tag, out, P, base_pval, seeds=SEEDS):
    """Tổng hợp ensemble, so mốc, lưu json — dùng chung cho mọi họ mô hình."""
    p_ens = np.mean([P[s] for s in seeds], axis=0)
    out['ensemble'] = gini_stability(w_val, y_val, p_ens)
    for k in ['gini_overall', 'stability', 'gap_fit_valid', 'best_iter']:
        v = np.array([out['seeds'][s][k] for s in seeds], dtype=float)
        out[f'{k}_mean_std'] = (float(v.mean()), float(v.std(ddof=1)))
    if base_pval is not None:
        mb = gini_stability(w_val, y_val, base_pval)
        dw = {w: out['ensemble']['weekly'][w] - mb['weekly'][w] for w in mb['weekly']}
        out['vs_base'] = {'d_valid_ens': out['ensemble']['gini_overall'] - mb['gini_overall'],
                          'd_stab_ens': out['ensemble']['stability'] - mb['stability'],
                          'weeks_won': int(sum(v > 0 for v in dw.values())),
                          'weekly_delta': {w: round(v, 4) for w, v in dw.items()}}
    with open(REPORTS / f'lgb08_{tag}.json', 'w', encoding='utf-8') as f:
        json.dump(out, f, indent=2, default=str)
    e = out['ensemble']
    print(f"\n[{tag}] per-seed Valid {out['gini_overall_mean_std'][0]:.5f} ± {out['gini_overall_mean_std'][1]:.5f} | "
          f"gap {out['gap_fit_valid_mean_std'][0]:.4f} | iter {out['best_iter_mean_std'][0]:.0f}")
    print(f"[{tag}] ensemble Valid {e['gini_overall']:.4f} | stab {e['stability']:.4f} "
          f"(mean {e['mean_gini']:.4f}, a {e['slope_a']:+.5f}, std_res {e['std_resid']:.4f})")
    if 'vs_base' in out:
        vb = out['vs_base']
        print(f"[{tag}] so mốc (ensemble): ΔValid {vb['d_valid_ens']:+.4f} | thắng {vb['weeks_won']}/8 | {vb['weekly_delta']}")
    return out, p_ens

def run_xgb(tag, Xf, Xe, Xv, names, ft, override=None, base_pval=None, seeds=SEEDS):
    override = override or {}
    dtr = xgb.QuantileDMatrix(Xf, label=y_fit, feature_names=names, feature_types=ft,
                              enable_categorical=True, max_bin=XGB_PARAMS['max_bin'])
    des = xgb.QuantileDMatrix(Xe, label=y_es, feature_names=names, feature_types=ft,
                              enable_categorical=True, ref=dtr)
    dva = xgb.DMatrix(Xv, feature_names=names, feature_types=ft, enable_categorical=True)
    out, P = {'tag': tag, 'family': 'xgboost', 'override': override, 'n_feats': len(names), 'seeds': {}}, {}
    for s in seeds:
        t0 = time.time()
        b = xgb.train({**XGB_PARAMS, **override, 'seed': s}, dtr, num_boost_round=NUM_BOOST_ROUND,
                      evals=[(des, 'es')], early_stopping_rounds=EARLY_STOPPING, verbose_eval=False)
        bi = b.best_iteration
        p = b.predict(dva, iteration_range=(0, bi + 1))
        m = gini_stability(w_val, y_val, p)
        gfit = float(2 * roc_auc_score(y_fit, b.predict(dtr, iteration_range=(0, bi + 1))) - 1)
        out['seeds'][s] = {'best_iter': int(bi), 'gini_fit': gfit, 'gap_fit_valid': gfit - m['gini_overall'],
                           **m, 'time_s': round(time.time() - t0, 1)}
        P[s] = p
        np.save(REPORTS / f'lgb08_{tag}_pval_s{s}.npy', p)
        b.save_model(str(MODELS / f'lgb08_{tag}_s{s}.json'))
        r = out['seeds'][s]
        print(f"[{tag}] seed {s} | iter {bi} | fit {gfit:.4f} | Valid {r['gini_overall']:.4f} | "
              f"gap {r['gap_fit_valid']:.4f} | stab {r['stability']:.4f} | {r['time_s']:.0f}s")
        del b; gc.collect()
    del dtr, des, dva; gc.collect()
    return finish(tag, out, P, base_pval, seeds)

EXP['xgb'], PENS['xgb'] = run_xgb('xgb', X_fit, X_es, X_val, FEAT_NAMES, FT_REF, base_pval=PENS['ref'])

# ---- Đa dạng & blend 50/50 (không học tham số trên Valid) ----
rho = spearmanr(PENS['ref'], PENS['xgb']).statistic
print(f'\nSpearman(ref LGB, XGB) trên Valid: {rho:.4f}')
p_blend = 0.5 * PENS['ref'] + 0.5 * PENS['xgb']
mb = gini_stability(w_val, y_val, p_blend)
mr = gini_stability(w_val, y_val, PENS['ref'])
dw = {w: mb['weekly'][w] - mr['weekly'][w] for w in mr['weekly']}
print(f"[blend lgb+xgb] Valid {mb['gini_overall']:.4f} | stab {mb['stability']:.4f} | "
      f"ΔValid so ref {mb['gini_overall'] - mr['gini_overall']:+.4f} | thắng {sum(v > 0 for v in dw.values())}/8 | "
      f"{ {w: round(v, 4) for w, v in dw.items()} }")
PENS['blend_lgb_xgb'] = p_blend

In [ ]:
def compare(p_new, p_base, label):
    mn, mb = gini_stability(w_val, y_val, p_new), gini_stability(w_val, y_val, p_base)
    dw = {w: mn['weekly'][w] - mb['weekly'][w] for w in mb['weekly']}
    won = sum(v > 0 for v in dw.values())
    d = mn['gini_overall'] - mb['gini_overall']
    print(f"[{label}] Valid {mn['gini_overall']:.4f} vs {mb['gini_overall']:.4f} | ΔValid {d:+.4f} | "
          f"Δstab {mn['stability'] - mb['stability']:+.4f} | thắng {won}/8 | "
          f"{'NHẬN' if (d > 0 and won >= 6) else 'không'} | { {w: round(v, 4) for w, v in dw.items()} }")
    return d, won

EXP['ref_s456'], PENS['ref_s456'] = run_lgb('ref_s456', X_fit, X_es, X_val, FEAT_NAMES, CAT_COLS,
                                            seeds=[45, 46, 47], base_pval=PENS['ref'])
PENS['lgb6'] = 0.5 * PENS['ref'] + 0.5 * PENS['ref_s456']

print()
compare(PENS['lgb6'], PENS['ref'], 'lgb6 vs ref (lgb3)')
compare(PENS['blend_lgb_xgb'], PENS['ref'], 'blend lgb3+xgb3 vs ref (lgb3)')
compare(PENS['blend_lgb_xgb'], PENS['lgb6'], 'blend lgb3+xgb3 vs lgb6  ← ĐỐI CHỨNG CHÍNH')
print(f"\nSpearman(ref, ref_s456): {spearmanr(PENS['ref'], PENS['ref_s456']).statistic:.4f} | "
      f"Spearman(ref, xgb): {spearmanr(PENS['ref'], PENS['xgb']).statistic:.4f}")

pl.DataFrame({'case_id': id_val, 'WEEK_NUM': w_val, 'target': y_val,
              **{f'pred_{k}': v for k, v in PENS.items()}}).write_parquet(REPORTS / 'lgb08_valid_preds_all.parquet')

In [ ]:
from scipy.stats import rankdata
ch = pl.read_parquet(REPORTS / 'champion_valid_preds.parquet')
assert np.array_equal(ch['case_id'].to_numpy(), id_val)
p_ch = ch['pred_champion'].to_numpy()
rk = lambda p: rankdata(p) / len(p)
print(f"Spearman(ref, Champion): {spearmanr(PENS['ref'], p_ch).statistic:.4f}  (ref–xgb 0.9916, ref–ref_s456 0.9940)")
for wt in [0.1, 0.2]:
    compare((1 - wt) * rk(PENS['ref']) + wt * rk(p_ch), PENS['ref'], f'ref + {wt:.1f}·Champion (rank)')

In [ ]:
try:
    import optuna
except ImportError:
    raise ImportError('Chạy trong terminal (venv): pip install optuna')
optuna.logging.set_verbosity(optuna.logging.INFO)

D_TR = lgb.Dataset(X_fit, label=y_fit, feature_name=FEAT_NAMES, categorical_feature=CAT_COLS,
                   params={'data_random_seed': 42, 'verbosity': -1, 'feature_pre_filter': False}, free_raw_data=False)
D_ES = lgb.Dataset(X_es, label=y_es, reference=D_TR, free_raw_data=False)

def prune_cb(trial, every=50):
    def _cb(env):
        if (env.iteration + 1) % every == 0:
            auc = [r[2] for r in env.evaluation_result_list if r[0] == 'es' and r[1] == 'auc'][0]
            trial.report(2 * auc - 1, env.iteration + 1)
            if trial.should_prune():
                raise optuna.TrialPruned()
    return _cb

def objective(trial):
    params = {**LGB_PARAMS,
              'num_leaves': trial.suggest_int('num_leaves', 15, 255, log=True),
              'min_child_samples': trial.suggest_int('min_child_samples', 50, 3000, log=True),
              'feature_fraction': trial.suggest_float('feature_fraction', 0.2, 1.0),
              'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
              'lambda_l1': trial.suggest_float('lambda_l1', 1e-3, 100.0, log=True),
              'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 100.0, log=True),
              'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 1.0),
              'path_smooth': trial.suggest_float('path_smooth', 0.0, 50.0),
              'cat_smooth': trial.suggest_float('cat_smooth', 1.0, 100.0, log=True),
              'extra_trees': trial.suggest_categorical('extra_trees', [False, True]),
              'bagging_seed': 42, 'feature_fraction_seed': 42}
    t0 = time.time()
    b = lgb.train(params, D_TR, num_boost_round=NUM_BOOST_ROUND, valid_sets=[D_ES], valid_names=['es'],
                  callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False), prune_cb(trial)])
    bi = b.best_iteration
    m = gini_stability(w_val, y_val, b.predict(X_val, num_iteration=bi))
    gfit = float(2 * roc_auc_score(y_fit, b.predict(X_fit, num_iteration=bi)) - 1)
    for k, v in {'best_iter': int(bi), 'es_gini': float(2 * b.best_score['es']['auc'] - 1), 'gini_fit': gfit,
                 'gap': gfit - m['gini_overall'], 'stability': m['stability'], 'slope_a': m['slope_a'],
                 'time_s': round(time.time() - t0, 1)}.items():
        trial.set_user_attr(k, v)
    del b; gc.collect()
    return m['gini_overall']

REF_TRIAL = {'num_leaves': 63, 'min_child_samples': 200, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
             'lambda_l1': 1e-3, 'lambda_l2': 10.0, 'min_gain_to_split': 0.0, 'path_smooth': 0.0,
             'cat_smooth': 10.0, 'extra_trees': False}

study = optuna.create_study(
    study_name='lgb08_tpe_v1', direction='maximize', load_if_exists=True,
    storage=f"sqlite:///{(MODELS / 'optuna_lgb08.db').as_posix()}",
    sampler=optuna.samplers.TPESampler(seed=42, multivariate=True, n_startup_trials=10),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=200, interval_steps=50))
if len(study.trials) == 0:
    study.enqueue_trial(REF_TRIAL)
study.optimize(objective, n_trials=30, timeout=4 * 3600, gc_after_trial=True)

In [ ]:
# ---- Tổng kết ----
tdf = study.trials_dataframe(attrs=('number', 'value', 'state', 'params', 'user_attrs', 'duration'))
tdf.to_csv(REPORTS / 'optuna_lgb08_trials.csv', index=False)
done = tdf[tdf['state'] == 'COMPLETE'].sort_values('value', ascending=False)
print(f"\nTrial: {len(tdf)} | COMPLETE {len(done)} | PRUNED {(tdf['state'] == 'PRUNED').sum()}")
print(f"Trial 0 (ref): {tdf.loc[tdf['number'] == 0, 'value'].values}")
cols = ['number', 'value', 'user_attrs_gap', 'user_attrs_best_iter', 'user_attrs_stability'] + \
       [c for c in done.columns if c.startswith('params_')]
print(done[cols].head(10).to_string(index=False))
with open(REPORTS / 'optuna_lgb08_best.json', 'w', encoding='utf-8') as f:
    json.dump({'best_value': study.best_value, 'best_params': study.best_params,
               'best_trial': study.best_trial.number, 'best_attrs': study.best_trial.user_attrs}, f, indent=2)

In [ ]:
def run_lgb(tag, Xf, Xe, Xv, names, cats, override=None, base_pval=None, seeds=SEEDS, save_model=True):
    override = override or {}
    dtr = lgb.Dataset(Xf, label=y_fit, feature_name=names, categorical_feature=cats,
                      params={'data_random_seed': 42, 'verbosity': -1}, free_raw_data=False)
    des = lgb.Dataset(Xe, label=y_es, reference=dtr, free_raw_data=False)
    out, P, IMP = {'tag': tag, 'override': override, 'n_feats': len(names), 'seeds': {}}, {}, []
    for s in seeds:
        t0 = time.time()
        params = {**LGB_PARAMS, **override, 'bagging_seed': s, 'feature_fraction_seed': s}
        b = lgb.train(params, dtr, num_boost_round=NUM_BOOST_ROUND, valid_sets=[des], valid_names=['es'],
                      callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False)])
        bi = b.best_iteration
        p = b.predict(Xv, num_iteration=bi)
        m = gini_stability(w_val, y_val, p)
        gfit = float(2 * roc_auc_score(y_fit, b.predict(Xf, num_iteration=bi)) - 1)
        out['seeds'][s] = {'best_iter': int(bi), 'gini_fit': gfit, 'gap_fit_valid': gfit - m['gini_overall'],
                           **m, 'time_s': round(time.time() - t0, 1)}
        P[s] = p
        IMP.append(b.feature_importance('gain', iteration=bi))
        np.save(REPORTS / f'lgb08_{tag}_pval_s{s}.npy', p)
        if save_model:
            b.save_model(str(MODELS / f'lgb08_{tag}_s{s}.txt'), num_iteration=bi)
        r = out['seeds'][s]
        print(f"[{tag}] seed {s} | iter {bi} | fit {gfit:.4f} | Valid {r['gini_overall']:.4f} | "
              f"gap {r['gap_fit_valid']:.4f} | stab {r['stability']:.4f} | {r['time_s']:.0f}s")
        del b; gc.collect()

    p_ens = np.mean([P[s] for s in seeds], axis=0)
    out['ensemble'] = gini_stability(w_val, y_val, p_ens)
    for k in ['gini_overall', 'stability', 'gap_fit_valid', 'best_iter']:
        v = np.array([out['seeds'][s][k] for s in seeds], dtype=float)
        out[f'{k}_mean_std'] = (float(v.mean()), float(v.std(ddof=1)))
    if base_pval is not None:
        mb = gini_stability(w_val, y_val, base_pval)
        dw = {w: out['ensemble']['weekly'][w] - mb['weekly'][w] for w in mb['weekly']}
        out['vs_base'] = {'d_valid_ens': out['ensemble']['gini_overall'] - mb['gini_overall'],
                          'd_stab_ens': out['ensemble']['stability'] - mb['stability'],
                          'weeks_won': int(sum(v > 0 for v in dw.values())),
                          'weekly_delta': {w: round(v, 4) for w, v in dw.items()}}
    g = np.mean([x / x.sum() for x in IMP], axis=0)
    pl.DataFrame({'feature': names, 'gain_share': g}).sort('gain_share', descending=True) \
      .write_csv(REPORTS / f'lgb08_{tag}_importance.csv')
    with open(REPORTS / f'lgb08_{tag}.json', 'w', encoding='utf-8') as f:
        json.dump(out, f, indent=2, default=str)

    e = out['ensemble']
    print(f"\n[{tag}] per-seed Valid {out['gini_overall_mean_std'][0]:.5f} ± {out['gini_overall_mean_std'][1]:.5f} | "
          f"gap {out['gap_fit_valid_mean_std'][0]:.4f} | iter {out['best_iter_mean_std'][0]:.0f}")
    print(f"[{tag}] ensemble Valid {e['gini_overall']:.4f} | stab {e['stability']:.4f} "
          f"(mean {e['mean_gini']:.4f}, a {e['slope_a']:+.5f}, std_res {e['std_resid']:.4f})")
    if 'vs_base' in out:
        vb = out['vs_base']
        print(f"[{tag}] so mốc (ensemble): ΔValid {vb['d_valid_ens']:+.4f} | Δstab {vb['d_stab_ens']:+.4f} | "
              f"thắng {vb['weeks_won']}/8 | {vb['weekly_delta']}")
    return out, p_ens

In [ ]:
BEST = json.load(open(REPORTS / 'optuna_lgb08_best.json', encoding='utf-8'))
print('Trial tốt nhất:', BEST['best_trial'], '| Valid 1 seed:', round(BEST['best_value'], 5))
OPT7 = BEST['best_params']
print(OPT7)

P_REF = np.mean([np.load(REPORTS / f'lgb08_ref_pval_s{s}.npy') for s in SEEDS], axis=0)
EXP, PENS = {}, {'ref': P_REF}
EXP['opt7'], PENS['opt7'] = run_lgb('opt7', X_fit, X_es, X_val, FEAT_NAMES, CAT_COLS,
                                    override=OPT7, base_pval=P_REF)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.INFO)

D_TR = lgb.Dataset(X_fit, label=y_fit, feature_name=FEAT_NAMES, categorical_feature=CAT_COLS,
                   params={'data_random_seed': 42, 'verbosity': -1, 'feature_pre_filter': False}, free_raw_data=False)
D_ES = lgb.Dataset(X_es, label=y_es, reference=D_TR, free_raw_data=False)

def prune_cb(trial, every=50):
    def _cb(env):
        if (env.iteration + 1) % every == 0:
            auc = [r[2] for r in env.evaluation_result_list if r[0] == 'es' and r[1] == 'auc'][0]
            trial.report(2 * auc - 1, env.iteration + 1)
            if trial.should_prune():
                raise optuna.TrialPruned()
    return _cb

In [ ]:
def objective2(trial):
    params = {**LGB_PARAMS,
              'num_leaves': trial.suggest_int('num_leaves', 8, 63, log=True),
              'min_child_samples': trial.suggest_int('min_child_samples', 300, 5000, log=True),
              'feature_fraction': trial.suggest_float('feature_fraction', 0.1, 0.6),
              'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
              'lambda_l1': trial.suggest_float('lambda_l1', 0.1, 50.0, log=True),
              'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 50.0, log=True),
              'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 0.5),
              'path_smooth': trial.suggest_float('path_smooth', 5.0, 100.0),
              'cat_smooth': trial.suggest_float('cat_smooth', 1.0, 100.0, log=True),
              'extra_trees': trial.suggest_categorical('extra_trees', [False, True]),
              'bagging_seed': 42, 'feature_fraction_seed': 42}
    t0 = time.time()
    b = lgb.train(params, D_TR, num_boost_round=NUM_BOOST_ROUND, valid_sets=[D_ES], valid_names=['es'],
                  callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False), prune_cb(trial, every=100)])
    bi = b.best_iteration
    m = gini_stability(w_val, y_val, b.predict(X_val, num_iteration=bi))
    gfit = float(2 * roc_auc_score(y_fit, b.predict(X_fit, num_iteration=bi)) - 1)
    for k, v in {'best_iter': int(bi), 'gini_fit': gfit, 'gap': gfit - m['gini_overall'],
                 'stability': m['stability'], 'time_s': round(time.time() - t0, 1)}.items():
        trial.set_user_attr(k, v)
    del b; gc.collect()
    return m['gini_overall']

OPT7 = json.load(open(REPORTS / 'optuna_lgb08_best.json', encoding='utf-8'))['best_params']
study2 = optuna.create_study(
    study_name='lgb08_tpe_v2', direction='maximize', load_if_exists=True,
    storage=f"sqlite:///{(MODELS / 'optuna_lgb08.db').as_posix()}",
    sampler=optuna.samplers.TPESampler(seed=7, multivariate=True, n_startup_trials=6),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1000, interval_steps=100))
if len(study2.trials) == 0:
    study2.enqueue_trial(OPT7)
study2.optimize(objective2, n_trials=15, timeout=2.5 * 3600, gc_after_trial=True)

In [ ]:
tdf2 = study2.trials_dataframe(attrs=('number', 'value', 'state', 'params', 'user_attrs'))
tdf2.to_csv(REPORTS / 'optuna_lgb08_v2_trials.csv', index=False)
done2 = tdf2[tdf2['state'] == 'COMPLETE'].sort_values('value', ascending=False)
print(f"Trial: {len(tdf2)} | COMPLETE {len(done2)} | PRUNED {(tdf2['state'] == 'PRUNED').sum()}")
print(f"Trial 0 (opt7): {tdf2.loc[tdf2['number'] == 0, 'value'].values}")
cols = ['number', 'value', 'user_attrs_gap', 'user_attrs_best_iter'] + [c for c in done2.columns if c.startswith('params_')]
print(done2[cols].head(8).to_string(index=False))
with open(REPORTS / 'optuna_lgb08_v2_best.json', 'w', encoding='utf-8') as f:
    json.dump({'best_value': study2.best_value, 'best_params': study2.best_params,
               'best_trial': study2.best_trial.number}, f, indent=2)